In [1]:
import pyspark
from pyspark.sql import SparkSession
import pyarrow.parquet as pq

In [2]:
spark = SparkSession.builder \
    .master("local[*]")\
    .appName('test')\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 18:31:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-05 02:48:18--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 99.84.245.9, 99.84.245.193, 99.84.245.141, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.245.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  23.6MB/s    in 14s     

2026-03-05 02:48:33 (21.4 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [3]:
df = spark.read \
    .option("header", "true") \
    .parquet('fhvhv_tripdata_2021-01.parquet')

In [6]:
df.show()

[Stage 2:=======================================>                   (2 + 1) / 3]

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [4]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N'),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 45, 56), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DO

In [5]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_re

In [17]:
#to avoid opening a big file we can get a portion of the file
#head_table = df.limit(100)

In [23]:
# Write to a new, smaller Parquet file using spark dataframe
#only use for very small datasets
df.limit(100).toPandas().to_parquet('head_fhvhv_tripdata_2021-01.parquet')

In [6]:
!wc -l head_fhvhv_tripdata_2021-01.parquet

83 head_fhvhv_tripdata_2021-01.parquet


In [7]:
#we can also use pandas to identify the types
import pandas as pd

In [8]:
df_pandas = pd.read_parquet('head_fhvhv_tripdata_2021-01.parquet')

In [9]:
df_pandas.dtypes

hvfhs_license_num                  str
dispatching_base_num               str
originating_base_num               str
request_datetime        datetime64[ns]
on_scene_datetime       datetime64[ns]
pickup_datetime         datetime64[ns]
dropoff_datetime        datetime64[ns]
PULocationID                     int64
DOLocationID                     int64
trip_miles                     float64
trip_time                        int64
base_passenger_fare            float64
tolls                          float64
bcf                            float64
sales_tax                      float64
congestion_surcharge           float64
airport_fee                    float64
tips                           float64
driver_pay                     float64
shared_request_flag                str
shared_match_flag                  str
access_a_ride_flag                 str
wav_request_flag                   str
wav_match_flag                     str
dtype: object

In [10]:
#create a spark dataframe from pandas
spark.createDataFrame(df_pandas).show()


[Stage 3:>                                                          (0 + 1) / 1]

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [11]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampType(), True), StructField('on_scene_datetime', TimestampType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag',

In [ ]:
#nteger takes 4 bytes, Long takes 8 bytes

In [12]:
#creating partitions
#Reading original file to create partitions
#df already exists, so we will continue using it
df = df.repartition(24)

In [13]:
df.write.parquet('fhvhv/2021/01')

In [55]:
#reading the parquets
df2 = spark.read.parquet('fhvhv/2021/01')

In [15]:
df2.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [23]:
#selecting information from dataframe
df2.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df2.hvfhs_license_num == 'HV0003')\
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-05 12:42:09|2021-01-05 13:04:44|          22|         261|
|2021-01-02 14:16:06|2021-01-02 14:22:35|         189|         181|
|2021-01-06 08:26:45|2021-01-06 08:53:00|          56|          75|
|2021-01-28 01:07:33|2021-01-28 01:17:39|         169|          69|
|2021-01-17 09:00:11|2021-01-17 09:13:05|         169|          69|
|2021-01-14 22:58:21|2021-01-14 23:12:56|         164|         145|
|2021-01-12 14:34:31|2021-01-12 15:01:36|         169|         250|
|2021-01-06 14:26:33|2021-01-06 14:44:08|          74|         152|
|2021-01-22 03:33:31|2021-01-22 03:42:22|          37|          80|
|2021-01-01 02:04:55|2021-01-01 02:11:25|         248|         213|
|2021-01-19 15:37:26|2021-01-19 16:02:07|          91|          14|
|2021-01-05 09:37:53|2021-01-05 10:09:26|       

In [25]:
#this is like head
df2.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID') \
    .filter(df2.hvfhs_license_num == 'HV0003')\
    .take(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 5, 12, 42, 9), dropoff_datetime=datetime.datetime(2021, 1, 5, 13, 4, 44), PULocationID=22, DOLocationID=261),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 14, 16, 6), dropoff_datetime=datetime.datetime(2021, 1, 2, 14, 22, 35), PULocationID=189, DOLocationID=181),
 Row(pickup_datetime=datetime.datetime(2021, 1, 6, 8, 26, 45), dropoff_datetime=datetime.datetime(2021, 1, 6, 8, 53), PULocationID=56, DOLocationID=75),
 Row(pickup_datetime=datetime.datetime(2021, 1, 28, 1, 7, 33), dropoff_datetime=datetime.datetime(2021, 1, 28, 1, 17, 39), PULocationID=169, DOLocationID=69),
 Row(pickup_datetime=datetime.datetime(2021, 1, 17, 9, 0, 11), dropoff_datetime=datetime.datetime(2021, 1, 17, 9, 13, 5), PULocationID=169, DOLocationID=69)]

In [27]:
#sql test
#SELECT * FROM df2 WHERE hvfhs_license_num = HV003

SyntaxError: invalid syntax (1182633075.py, line 2)

In [49]:
#creating a user defined function in sparks UDF
from pyspark.sql import functions as F
from pyspark.sql import types

In [39]:
def verify_id(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f"s/{num:03x}"
    elif num % 3 == 0:
        return f"a/{num:03x}"
    else:
        return f"e/{num:03x}"

In [40]:
verify_id('B02884')

's/b44'

In [50]:
verify_id_udf = F.udf(verify_id, returnType= types.StringType())

In [56]:
#Adding two new columns to the spark dataframe
df2 = df2 \
    .withColumn('pickup_date', F.to_date(df2.pickup_datetime))\
    .withColumn('dropoff_date', F.to_date(df2.dropoff_datetime)) \
    .withColumn('base_id', verify_id_udf(df2.dispatching_base_num))\
    .select('base_id','pickup_date','dropoff_date','PULocationID','DOLocationID')

In [58]:
#gives error because the table has not been written???
#explanation: In your previous cell (Cell [51]), you added the pickup_date and dropoff_date columns using .withColumn(). However, Spark DataFrames are immutable, 
#meaning they cannot be changed in place. When you run .withColumn(), it doesn't modify the existing df2; it returns a new DataFrame that contains those new columns.
#How to fix it: You need to reassign the result of your transformations to the variable df2.example df2 = df2 \...
df2.select('base_id','pickup_date','dropoff_date','PULocationID','DOLocationID') \
    .filter(df2.pickup_date.isNotNull())\
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/acc| 2021-01-05|  2021-01-05|          22|         261|
|  e/b33| 2021-01-02|  2021-01-02|         189|         181|
|  s/acd| 2021-01-06|  2021-01-06|          56|          75|
|  s/acd| 2021-01-28|  2021-01-28|         169|          69|
|  e/b3b| 2021-01-17|  2021-01-17|         169|          69|
|  e/9ce| 2021-01-29|  2021-01-29|         198|          37|
|  e/acc| 2021-01-14|  2021-01-14|         164|         145|
|  e/9ce| 2021-01-10|  2021-01-10|          58|         213|
|  e/9ce| 2021-01-21|  2021-01-21|          51|         174|
|  e/95b| 2021-01-12|  2021-01-12|         169|         250|
|  e/b47| 2021-01-06|  2021-01-06|          74|         152|
|  e/9ce| 2021-01-16|  2021-01-16|         145|         138|
|  e/b30| 2021-01-22|  2021-01-22|          37|          80|
|  s/b44| 2021-01-01|  2